# Skeleton for using jupyter on UTE Cloud Testlines

The below code is used, to determine the current testline configuration by the host name.

With the host name information, the testline configuration is loaded, so you can later on
work with the common keywords on the testline.

You need to execute the cell (by pressing [Ctrl]+[Enter] on the cell or pressing the play button)

After running the cell, an ip address is printed. If the ip address corresponds to the address of the local machine,
the correct testline configuration is loaded.

In [ ]:
*** Settings ***
Library    jupyter_helper.py
Resource    ../ute_workarounds.robot

*** Test Cases ***
Load TL Config
    Load Local Testline Config
    UTE Cloud Workaround
    Log    HostPC is ${tl.test_pc}
    ${host_ip}=    Get Host IP
    ${local_ip}=   Set Variable   ${tl.test_pc.default_ssh_connection_details.addr}
    Should Be Equal    ${host_ip}   ${local_ip}    The tl config IP ${local_ip} does not fit the host ip ${host_ip}
    
    Set Variable   ${local_ip}
    

After having the testline configuration loaded with the first cell on this page, you can start interacting with your testline. 

The next cell is using the keywords from ute_workarounds to bring the testline in the OnAir status

After executing this Keyword, you are able to proceed with further interactions, like attaching UES, collecting logs, ...

# Test Setup and Teardown to connect to running egate and edaemon

Since Jupyter is instanciating the ute_wtssim again again, with each start of a new cell, the alias from a previous run cell is no more valid. A workaround for this is to define a test setup and a test teardown to setup the utw_wtssim with each test case, and tear it down after each test case.

**Don't run the keywords directly, this will lead to errors.**

In [ ]:
*** Settings ***
Library     ute_wtssim
Resource    ../ute_workarounds.robot
Test Setup      Connect to running egate
Test Teardown   Teardown the wts sim
    

*** Keywords ***
Connect to running egate
    ${wts_alias}=      Set Variable    ap_egate
    ${egate_port}=     Set Variable    ${tl.core.egate_port}
    ${edaemon_alias}=  Set Variable    ap_edaemon
    ${edaemon_port}=   Set Variable    ${tl.core.edaemon_port}
    Set Suite Variable    ${wts_alias}
    Set Suite Variable    ${edaemon_alias}

    ute_wtssim.setup_egate    egate_port=${egate_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${wts_alias}
    ute_wtssim.setup_edaemon    edaemon_port=${edaemon_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${edaemon_alias}
    ${wts_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_info    pass_regexp=Process ID    wts_alias=${wts_alias}
    ${sim_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_state_disp    pass_regexp=The state is ACTIVE    wts_alias=${wts_alias}
    
    Run Keyword If    $wts_status == False or $sim_status == False    Fail   Unable To Connect to running Egate

Teardown the wts sim
    ute_wtssim.teardown_egate     alias=${wts_alias}
    ute_wtssim.teardown_edaemon   alias=${edaemon_alias}


In [ ]:
%%python module JupyterDisplay

from IPython.display import display_pretty, Markdown, display, Code, JSON
import sys

sys.path.append('/home/ute/DEVELOPMENT/robotws')

def jupyter_display_markdown(text):
    display(Markdown(text))
    
def jupyter_display_json(output_object):
    display(JSON(output_object))

def jupyter_display_text(text):
    display(Code(text.decode('utf-8')))


In [ ]:
*** Settings ***
Resource   ../testline.robot
Library    JupyterDisplay

*** Test Cases ***
Get the gNodeB SW Version
    ${sw_version}=    Get gNB SW version
    Set Suite Variable    ${sw_version}
    Display    gNodeB Software Version is [${sw_version}]

In [ ]:
*** Settings ***
Library    ../python/MCSharkDissectors.py

*** Test Cases ***
Check If the installation of dissectors is working well
    Install MCShark Dissector with script   ${sw_version}